# Research Workbench: Market Structure & Supply/Demand Validation

This notebook is the permanent development environment for market-structure research in the Forex_DNN project. It serves as a workbench for validating deterministic algorithms, inspecting feature engineering, and generating labeled datasets for machine learning.

**Primary Goals:**
- Visual validation of Swing High/Low, BOS, and CHOCH detection.
- Visual validation of Supply and Demand zone lifecycles.
- Direct comparison between visual structure and numerical feature vectors.
- Human annotation of events for ML training.

## Setup and Imports

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from datetime import datetime, timezone, timedelta
import logging

# Ensure project root is in path
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'examples':
    project_root = os.path.abspath(os.path.join(current_dir, '..'))
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Framework Imports
from Collecting_Data.indicators import IndicatorEngine
from MarketStructure.market_structure import MarketStructureEngine
from MarketStructure.supply_demand import SupplyDemandEngine

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('ResearchWorkbench')

### SECTION 1 — Configuration

In [ ]:
SYMBOL = "EURUSD_o"
TIMEFRAME = "M5"
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"
WINDOW_SIZE = 300
LOOKBACK = 3
SHOW_ONLY_EVENTS = False
SAVE_IMAGES = False
EXPORT_RESULTS = True

DATA_DIRECTORY = os.path.join(project_root, "Data")
VALIDATION_DIRECTORY = os.path.join(project_root, "Validation")
os.makedirs(VALIDATION_DIRECTORY, exist_ok=True)
os.makedirs(os.path.join(VALIDATION_DIRECTORY, "charts"), exist_ok=True)
os.makedirs(os.path.join(VALIDATION_DIRECTORY, "events"), exist_ok=True)

### SECTION 2 — Load Historical Data

We load CSV files produced by the framework. If no data is found, we generate a synthetic dataset for demonstration.

In [ ]:
def generate_synthetic_data(n=2000):
    logger.info("Generating synthetic data for demonstration...")
    np.random.seed(42)
    dates = pd.date_range(start="2024-01-01", periods=n, freq="5min", tz="UTC")
    
    # Create a trending price with some noise and swings
    trend = np.linspace(1.0800, 1.1000, n)
    cycle = 0.0020 * np.sin(np.linspace(0, 10 * np.pi, n))
    noise = np.random.normal(0, 0.0002, n)
    close = trend + cycle + noise
    
    df = pd.DataFrame({
        'Datetime': dates,
        'Open': close + np.random.normal(0, 0.0001, n),
        'High': close + np.abs(np.random.normal(0.0003, 0.0001, n)),
        'Low': close - np.abs(np.random.normal(0.0003, 0.0001, n)),
        'Close': close,
        'TickVolume': np.random.randint(100, 1000, n),
        'Spread': np.random.randint(1, 5, n)
    })
    # Ensure High is highest and Low is lowest
    df['High'] = df[['Open', 'Close', 'High']].max(axis=1)
    df['Low'] = df[['Open', 'Close', 'Low']].min(axis=1)
    return df

data_path = os.path.join(DATA_DIRECTORY, f"{SYMBOL}_{TIMEFRAME}.csv")
if os.path.exists(data_path):
    df_raw = pd.read_csv(data_path, parse_dates=['Datetime'])
    logger.info(f"Loaded {len(df_raw)} rows from {data_path}")
else:
    df_raw = generate_synthetic_data()

def validate_data(df):
    report = []
    
    # Duplicates
    dupes = df['Datetime'].duplicated().sum()
    report.append(("Duplicates", dupes))
    
    # Ordering
    is_ordered = df['Datetime'].is_monotonic_increasing
    report.append(("Timestamp Ordering", "OK" if is_ordered else "FAIL"))
    
    # NaNs
    nans = df.isnull().sum().sum()
    report.append(("NaNs", nans))
    
    # Gaps
    diffs = df['Datetime'].diff().dt.total_seconds()
    expected_diff = (df['Datetime'].iloc[1] - df['Datetime'].iloc[0]).total_seconds()
    gaps = (diffs > expected_diff).sum()
    report.append(("Missing Candles (Gaps)", gaps))
    
    # Impossible Values
    impossible = ((df['High'] < df['Low']) | (df['High'] < df['Open']) | (df['High'] < df['Close']) | 
                  (df['Low'] > df['Open']) | (df['Low'] > df['Close'])).sum()
    report.append(("Impossible OHLC", impossible))
    
    # Metrics
    report.append(("Rows", len(df)))
    report.append(("Date Range", f"{df['Datetime'].min()} to {df['Datetime'].max()}"))
    report.append(("Memory Usage", f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"))
    
    # Statistics
    report.append(("Average Spread", f"{df['Spread'].mean():.2f}"))
    tr = pd.concat([df['High'] - df['Low'], (df['High'] - df['Close'].shift(1)).abs(), (df['Low'] - df['Close'].shift(1)).abs()], axis=1).max(axis=1)
    report.append(("Average ATR (14)", f"{tr.rolling(14).mean().mean():.6f}"))
    
    print("--- DATA VALIDATION REPORT ---")
    for k, v in report:
        print(f"{k:<25}: {v}")

validate_data(df_raw)

### SECTION 3 — Indicator Validation

Run IndicatorEngine to generate baseline technical features.

In [ ]:
ie = IndicatorEngine(ema_periods=[50, 600])
df_indicators = ie.calculate(df_raw)

print(f"Generated Columns: {list(df_indicators.columns[len(df_raw.columns):])}")
print(f"EMA 50 Available: {'ema_50' in df_indicators.columns}")
print(f"EMA 600 Available: {'ema_600' in df_indicators.columns}")
print(f"ATR Available: {'atr_14' in df_indicators.columns}")
print(f"EMA Slope Available: {'ema_slope_600' in df_indicators.columns}")
print(f"Indicator NaN Count: {df_indicators.isna().sum().sum()}")

### SECTION 4 — MarketStructureEngine Validation

Run MarketStructureEngine to detect swings, BOS, and CHOCH.

In [ ]:
mse = MarketStructureEngine(lookback=LOOKBACK)
df_structure = mse.process(df_indicators)

stats_mse = mse.get_summary(df_structure)

print("--- MARKET STRUCTURE STATISTICS ---")
print(f"Total Swing Highs: {len([s for s in mse.swings if s.type == 'High'])}")
print(f"Total Swing Lows: {len([s for s in mse.swings if s.type == 'Low'])}")
print(f"Bullish BOS: {len([b for b in mse.bos_list if b.direction == 1])}")
print(f"Bearish BOS: {len([b for b in mse.bos_list if b.direction == -1])}")
print(f"Bullish CHOCH: {len([c for c in mse.choch_list if c.new_trend == 1])}")
print(f"Bearish CHOCH: {len([c for c in mse.choch_list if c.new_trend == -1])}")

trends = df_structure['trend'].value_counts()
longest_trend = 0
curr_trend = 0
curr_val = 0
for val in df_structure['trend']:
    if val != 0 and val == curr_val:
        curr_trend += 1
        longest_trend = max(longest_trend, curr_trend)
    else:
        curr_val = val
        curr_trend = 1 if val != 0 else 0
print(f"Longest Trend: {longest_trend} bars")

bos_distances = [b.index - mse.bos_list[i-1].index for i, b in enumerate(mse.bos_list) if i > 0]
if bos_distances:
    print(f"Average bars between BOS: {np.mean(bos_distances):.2f}")
    print(f"Median bars between BOS: {np.median(bos_distances):.2f}")
    print(f"Max bars between BOS: {np.max(bos_distances)}")

impulses = [b.distance for b in mse.bos_list]
if impulses:
    print(f"Average Impulse: {np.mean(impulses):.6f}")
    print(f"Maximum Impulse: {np.max(impulses):.6f}")

choch_distances = [c.index - mse.choch_list[i-1].index for i, c in enumerate(mse.choch_list) if i > 0]
if choch_distances:
    print(f"Average bars between CHOCH: {np.mean(choch_distances):.2f}")

print(f"Current Trend: {stats_mse['structure_state']} ({stats_mse['trend']})")

### SECTION 5 — SupplyDemandEngine Validation

Run SupplyDemandEngine to detect institutional zones.

In [ ]:
sde = SupplyDemandEngine()
df_final = sde.process(df_structure)

zones = sde.zones
print("--- SUPPLY & DEMAND STATISTICS ---")
print(f"Total Supply Zones: {len([z for z in zones if z.type == 'Supply'])}")
print(f"Total Demand Zones: {len([z for z in zones if z.type == 'Demand'])}")
print(f"Broken Zones: {len([z for z in zones if z.broken])}")
print(f"Mitigated Zones: {len([z for z in zones if z.mitigated])}")
print(f"Active (Unbroken) Zones: {len([z for z in zones if not z.broken])}")
print(f"Fresh Zones: {len([z for z in zones if z.freshness])}")

if zones:
    print(f"Average Width: {np.mean([z.width for z in zones]):.6f}")
    print(f"Average Strength: {np.mean([z.strength_score for z in zones]):.2f}")
    
    lifetimes = [z.broken_idx - z.created_idx for z in zones if z.broken]
    if lifetimes:
        print(f"Average Lifetime (Broken Zones): {np.mean(lifetimes):.2f} bars")

## VISUALIZATION SUITE

In [ ]:
def plot_market_structure(df, start_idx=0, end_idx=500, title="Market Structure Validation"):
    subset = df.iloc[start_idx:end_idx].copy()
    
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Price
    ax.plot(subset.index, subset['Close'], color='black', alpha=0.3, label='Price')
    
    # EMAs
    if 'ema_50' in subset.columns:
        ax.plot(subset.index, subset['ema_50'], color='blue', alpha=0.5, label='EMA 50')
    if 'ema_600' in subset.columns:
        ax.plot(subset.index, subset['ema_600'], color='red', alpha=0.5, label='EMA 600')
    
    # Swings
    swings = [s for s in mse.swings if start_idx <= s.index < end_idx]
    highs = [s for s in swings if s.type == 'High']
    lows = [s for s in swings if s.type == 'Low']
    
    ax.scatter([s.index for s in highs], [s.price for s in highs], marker='^', color='green', s=100, label='Swing High')
    ax.scatter([s.index for s in lows], [s.price for s in lows], marker='v', color='red', s=100, label='Swing Low')
    
    # BOS
    bos = [b for b in mse.bos_list if start_idx <= b.index < end_idx]
    bull_bos = [b for b in bos if b.direction == 1]
    bear_bos = [b for b in bos if b.direction == -1]
    
    ax.scatter([b.index for b in bull_bos], [subset.loc[b.index, 'High'] for b in bull_bos], marker='$\\uparrow$', color='green', s=150, label='Bull BOS')
    ax.scatter([b.index for b in bear_bos], [subset.loc[b.index, 'Low'] for b in bear_bos], marker='$\\downarrow$', color='red', s=150, label='Bear BOS')
    
    # CHOCH
    chochs = [c for c in mse.choch_list if start_idx <= c.index < end_idx]
    bull_choch = [c for c in chochs if c.new_trend == 1]
    bear_choch = [c for c in chochs if c.new_trend == -1]
    
    ax.scatter([c.index for c in bull_choch], [subset.loc[c.index, 'High'] for c in bull_choch], marker='o', color='green', s=100, facecolors='none', label='Bull CHOCH')
    ax.scatter([c.index for c in bear_choch], [subset.loc[c.index, 'Low'] for c in bear_choch], marker='o', color='red', s=100, label='Bear CHOCH')
    
    ax.set_title(title)
    ax.set_xlabel("Bar Index")
    ax.set_ylabel("Price")
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_supply_demand(df, start_idx=0, end_idx=500, title="Supply & Demand Validation"):
    subset = df.iloc[start_idx:end_idx].copy()
    fig, ax = plt.subplots(figsize=(15, 8))
    
    ax.plot(subset.index, subset['Close'], color='black', alpha=0.5)
    
    # Filter zones that were active or relevant during this window
    relevant_zones = [z for z in sde.zones if z.created_idx < end_idx and (not z.broken or z.broken_idx >= start_idx)]
    
    for z in relevant_zones:
        color = 'red' if z.type == 'Supply' else 'blue'
        alpha = 0.2
        if z.broken:
            color = 'gray'
            alpha = 0.1
        elif not z.freshness:
            alpha = 0.3 # Mitigated but not broken
        
        draw_start = max(start_idx, z.created_idx)
        draw_end = min(end_idx, z.broken_idx if z.broken else end_idx)
        
        rect = patches.Rectangle((draw_start, z.lower), draw_end - draw_start, z.upper - z.lower, 
                                 linewidth=1, edgecolor=color, facecolor=color, alpha=alpha)
        ax.add_patch(rect)
        
    ax.set_title(title)
    ax.set_xlabel("Bar Index")
    ax.set_ylabel("Price")
    ax.grid(True, alpha=0.3)
    plt.show()

def plot_combined(df, start_idx=0, end_idx=500, title="Combined Debug Chart"):
    subset = df.iloc[start_idx:end_idx].copy()
    fig, ax = plt.subplots(figsize=(18, 10))
    
    # 1. Price and EMAs
    ax.plot(subset.index, subset['Close'], color='black', alpha=0.4)
    if 'ema_50' in subset.columns: ax.plot(subset.index, subset['ema_50'], color='blue', alpha=0.3)
    if 'ema_600' in subset.columns: ax.plot(subset.index, subset['ema_600'], color='red', alpha=0.3)
    
    # 2. Zones
    relevant_zones = [z for z in sde.zones if z.created_idx < end_idx and (not z.broken or z.broken_idx >= start_idx)]
    for z in relevant_zones:
        color = 'red' if z.type == 'Supply' else 'blue'
        alpha = 0.15
        if z.broken: color = 'gray'; alpha = 0.05
        draw_start = max(start_idx, z.created_idx)
        draw_end = min(end_idx, z.broken_idx if z.broken else end_idx)
        rect = patches.Rectangle((draw_start, z.lower), draw_end - draw_start, z.upper - z.lower, facecolor=color, alpha=alpha)
        ax.add_patch(rect)

    # 3. Market Structure
    swings = [s for s in mse.swings if start_idx <= s.index < end_idx]
    ax.scatter([s.index for s in swings if s.type == 'High'], [s.price for s in swings if s.type == 'High'], marker='^', color='green', s=60)
    ax.scatter([s.index for s in swings if s.type == 'Low'], [s.price for s in swings if s.type == 'Low'], marker='v', color='red', s=60)
    
    bos = [b for b in mse.bos_list if start_idx <= b.index < end_idx]
    ax.scatter([b.index for b in bos if b.direction == 1], [subset.loc[b.index, 'High'] for b in bos if b.direction == 1], marker='$\\uparrow$', color='green', s=100)
    ax.scatter([b.index for b in bos if b.direction == -1], [subset.loc[b.index, 'Low'] for b in bos if b.direction == -1], marker='$\\downarrow$', color='red', s=100)
    
    chochs = [c for c in mse.choch_list if start_idx <= c.index < end_idx]
    ax.scatter([c.index for c in chochs if c.new_trend == 1], [subset.loc[c.index, 'High'] for c in chochs if c.new_trend == 1], marker='o', color='green', s=80, facecolors='none')
    ax.scatter([c.index for c in chochs if c.new_trend == -1], [subset.loc[c.index, 'Low'] for c in chochs if c.new_trend == -1], marker='o', color='red', s=80)

    ax.set_title(title)
    ax.grid(True, alpha=0.2)
    plt.show()

plot_combined(df_final, 0, WINDOW_SIZE)

## EventNavigator and Interactive Workbench

In [ ]:
class EventNavigator:
    def __init__(self, df, mse, sde):
        self.df = df
        self.mse = mse
        self.sde = sde
        self.events = self._index_events()
        self.current_idx = 0
        self.annotations_path = os.path.join(VALIDATION_DIRECTORY, "annotations.csv")
        
    def _index_events(self):
        evs = []
        for s in self.mse.swings: evs.append({'id': len(evs), 'type': f'Swing {s.type}', 'idx': s.index, 'price': s.price})
        for b in self.mse.bos_list: evs.append({'id': len(evs), 'type': 'Bull BOS' if b.direction == 1 else 'Bear BOS', 'idx': b.index, 'price': b.broken_level})
        for c in self.mse.choch_list: evs.append({'id': len(evs), 'type': 'Bull CHOCH' if c.new_trend == 1 else 'Bear CHOCH', 'idx': c.index, 'price': c.price})
        for z in self.sde.zones: evs.append({'id': len(evs), 'type': f'{z.type} Created', 'idx': z.created_idx, 'price': z.mid})
        return sorted(evs, key=lambda x: x['idx'])
    
    def list_events(self, event_type=None, start=None, end=None):
        filtered = self.events
        if event_type: filtered = [e for e in filtered if event_type in e['type']]
        if start: filtered = [e for e in filtered if e['idx'] >= start]
        if end: filtered = [e for e in filtered if e['idx'] <= end]
        return pd.DataFrame(filtered)

    def show_event(self, event_id=None, event_type=None, number=None):
        if event_id is not None:
            event = next((e for e in self.events if e['id'] == event_id), None)
        elif event_type is not None and number is not None:
            matches = [e for e in self.events if event_type in e['type']]
            event = matches[number] if number < len(matches) else None
        else:
            event = self.events[self.current_idx]
            
        if event:
            print(f"--- EVENT REPORT: {event['type']} (ID: {event['id']}) ---")
            print(f"Timestamp: {self.df.loc[event['idx'], 'Datetime']}")
            print(f"Price: {event['price']:.6f}")
            self.show_feature_vector(event['idx'])
            plot_combined(self.df, max(0, event['idx'] - 75), min(len(self.df), event['idx'] + 75), title=f"Event: {event['type']}")
            
    def next_event(self, event_type=None):
        for i in range(self.current_idx + 1, len(self.events)):
            if event_type is None or event_type in self.events[i]['type']:
                self.current_idx = i
                self.show_event()
                return
        print("No more events found.")
        
    def previous_event(self, event_type=None):
        for i in range(self.current_idx - 1, -1, -1):
            if event_type is None or event_type in self.events[i]['type']:
                self.current_idx = i
                self.show_event()
                return
        print("Already at the first event.")

    def show_date(self, date_str):
        target = pd.to_datetime(date_str).tz_localize('UTC')
        idx = (self.df['Datetime'] - target).abs().idxmin()
        plot_combined(self.df, max(0, idx - 100), min(len(self.df), idx + 100), title=f"Date Jump: {date_str}")
        
    def show_feature_vector(self, idx):
        row = self.df.iloc[idx]
        features = row.to_dict()
        # Simplify for display
        display_feats = {k: v for k, v in features.items() if not isinstance(v, pd.Timestamp)}
        print("\n--- FEATURE VECTOR PREVIEW ---")
        print(pd.Series(display_feats).to_frame(name='Value'))
        
    def mark_correct(self, event_id, note=""):
        self._annotate(event_id, "CORRECT", note)
        
    def mark_incorrect(self, event_id, note=""):
        self._annotate(event_id, "INCORRECT", note)

    def mark_uncertain(self, event_id, note=""):
        self._annotate(event_id, "UNCERTAIN", note)

    def annotate(self, event_id, note):
        self._annotate(event_id, "ANNOTATED", note)
        
    def _annotate(self, event_id, status, note):
        event = next((e for e in self.events if e['id'] == event_id), None)
        new_row = {
            'event_id': event_id,
            'event_type': event['type'] if event else 'Unknown',
            'timestamp': self.df.loc[event['idx'], 'Datetime'] if event else datetime.now(),
            'status': status,
            'note': note
        }
        df_ann = pd.DataFrame([new_row])
        df_ann.to_csv(self.annotations_path, mode='a', header=not os.path.exists(self.annotations_path), index=False)
        print(f"Annotation saved for Event {event_id}: {status}")

    def show_zone(self, zone_id):
        z = self.sde.zones[zone_id] if zone_id < len(self.sde.zones) else None
        if z:
            print(f"--- ZONE REPORT: {z.type} (ID: {zone_id}) ---")
            print(f"Created: {z.created_time} (Idx: {z.created_idx})")
            print(f"Strength: {z.strength_score:.2f} | Touches: {z.touch_count} | Fresh: {z.freshness}")
            if z.broken: print(f"Broken at Idx: {z.broken_idx}")
            plot_supply_demand(self.df, max(0, z.created_idx - 50), min(len(self.df), (z.broken_idx or z.created_idx) + 100))

    def save_event(self, event_id):
        event = next((e for e in self.events if e['id'] == event_id), None)
        if not event: return
        
        path = os.path.join(VALIDATION_DIRECTORY, "events", f"event_{event_id:03d}.png")
        # Using a non-interactive backend for saving if needed, but usually plt.savefig works
        plt.ioff()
        fig, ax = plt.subplots(figsize=(18, 10))
        # Redraw logic simplified for saving
        start_idx = max(0, event['idx'] - 75)
        end_idx = min(len(self.df), event['idx'] + 75)
        # Re-using plot logic would be better but keeping it contained for now
        subset = self.df.iloc[start_idx:end_idx]
        ax.plot(subset.index, subset['Close'], color='black', alpha=0.4)
        plt.title(f"Event: {event['type']} (ID: {event_id})")
        plt.savefig(path)
        plt.close(fig)
        plt.ion()
        print(f"Event {event_id} chart saved to {path}")

    def compare_events(self, ids):
        for eid in ids:
            self.show_event(eid)

    def show_structure(self, start_idx, end_idx):
        filtered = [e for e in self.events if start_idx <= e['idx'] <= end_idx and ('BOS' in e['type'] or 'CHOCH' in e['type'])]
        print(f"--- STRUCTURE TIMELINE ({start_idx} to {end_idx}) ---")
        for e in filtered:
            print(f"Index {e['idx']}: {e['type']} at {e['price']:.6f}")
        plot_combined(self.df, start_idx, end_idx, title="Structure Timeline")

    def review_events(self, event_type=None):
        print(f"--- Reviewing {event_type or 'All'} Events ---")
        print("Type 'exit' to stop, anything else for next event.")
        filtered = [e for e in self.events if event_type is None or event_type in e['type']]
        for e in filtered:
            self.show_event(e['id'])
            user_input = input("Press ENTER for next event (or type 'exit'): ")
            if user_input.lower() == 'exit': break

nav = EventNavigator(df_final, mse, sde)
nav.show_event(0)

### Validation Checks

Detect suspicious situations in the data.

In [ ]:
def run_validation_checks(df, mse, sde):
    warnings = []
    
    # 1. Duplicate BOS
    for i in range(1, len(mse.bos_list)):
        if mse.bos_list[i].index == mse.bos_list[i-1].index:
            warnings.append(f"Duplicate BOS at index {mse.bos_list[i].index}")
            
    # 2. CHOCH followed by opposite CHOCH immediately
    for i in range(1, len(mse.choch_list)):
        if mse.choch_list[i].index - mse.choch_list[i-1].index < 5:
            warnings.append(f"Rapid CHOCH reversal at index {mse.choch_list[i].index}")
            
    # 3. Zone width check
    for z in sde.zones:
        if z.width <= 0:
            warnings.append(f"Invalid zone width at index {z.created_idx}")
            
    if warnings:
        print("--- VALIDATION WARNINGS ---")
        for w in warnings:
            print(f"[WARNING] {w}")
    else:
        print("No structural anomalies detected.")

run_validation_checks(df_final, mse, sde)

### Export Results

In [ ]:
if EXPORT_RESULTS:
    df_final.to_csv(os.path.join(VALIDATION_DIRECTORY, "market_analysis.csv"), index=False)
    nav.list_events().to_csv(os.path.join(VALIDATION_DIRECTORY, "events.csv"), index=False)
    
    # Generate simple summary report
    with open(os.path.join(VALIDATION_DIRECTORY, "validation_report.txt"), "w") as f:
        f.write(f"Validation Report - {SYMBOL} {TIMEFRAME}\n")
        f.write(f"Date: {datetime.now()}\n")
        f.write(f"Total Bars: {len(df_final)}\n")
        f.write(f"BOS Count: {len(mse.bos_list)}\n")
        f.write(f"CHOCH Count: {len(mse.choch_list)}\n")
        f.write(f"Zones Detected: {len(sde.zones)}\n")
    
    print(f"Results exported to {VALIDATION_DIRECTORY}")

### ADDITIONAL ANALYSIS CHARTS

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12))

# Bars Since BOS
axes[0, 0].plot(df_final.index[:WINDOW_SIZE], df_final['bars_since_bos'][:WINDOW_SIZE])
axes[0, 0].set_title("Bars Since BOS")
axes[0, 0].grid(True, alpha=0.3)

# Bars Since CHOCH
axes[0, 1].plot(df_final.index[:WINDOW_SIZE], df_final['bars_since_choch'][:WINDOW_SIZE])
axes[0, 1].set_title("Bars Since CHOCH")
axes[0, 1].grid(True, alpha=0.3)

# Distance to Supply
axes[1, 0].plot(df_final.index[:WINDOW_SIZE], df_final['nearest_supply_distance'][:WINDOW_SIZE])
axes[1, 0].set_title("Distance to Nearest Supply")
axes[1, 0].grid(True, alpha=0.3)

# Distance to Demand
axes[1, 1].plot(df_final.index[:WINDOW_SIZE], df_final['nearest_demand_distance'][:WINDOW_SIZE])
axes[1, 1].set_title("Distance to Nearest Demand")
axes[1, 1].grid(True, alpha=0.3)

# Strength Histogram
strengths = [z.strength_score for z in sde.zones]
axes[2, 0].hist(strengths, bins=20, color='purple', alpha=0.7)
axes[2, 0].set_title("Supply/Demand Strength Histogram")
axes[2, 0].grid(True, alpha=0.3)

# Empty/Timeline placeholder
axes[2, 1].axis('off')
axes[2, 1].text(0.5, 0.5, 'Structure Timeline Browser\nAvailable via EventNavigator', ha='center', va='center')

plt.tight_layout()
plt.show()